In [109]:
from datetime import datetime, timedelta, date
import requests
import time
import pandas as pd
import holidays
from category_encoders import TargetEncoder
import pickle
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import xgboost as xgb
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# Pipeline opstellen

## Ophalen data

In [50]:
def ophalenKijkcijferData(startDate, endDate):
  print(f"Ophalen kijkcijfer-data van {startDate.year}-{startDate.month}-{startDate.day} tot {endDate.year}-{endDate.month}-{endDate.day}")
  kijkcijfersData = []
  #elke dag ophalen (startDate is huidige dag)
  while startDate <= endDate:
    datum = f"{startDate.year}-{startDate.month}-{startDate.day}"
    url = f"https://api.cim.be/api/cim_tv_public_results_daily_views?dateDiff={datum}&reportType=north"

    try:
      response = requests.get(url)
      if response.status_code == 200:
        data = response.json()
        programmaLijst = data.get('hydra:member', [])
                        
        for programma in programmaLijst:
          try:
            kijkcijfersData.append({
              'dateDiff': programma.get('dateDiff'),
              'ranking': programma.get('ranking'),
              'description': programma.get('description'),
              'channel': programma.get('channel'),
              'startTime': programma.get('startTime'),
              'rLength': programma.get('rLength'),
              'rateInK': programma.get('rateInK'),
              'live': programma.get('live')
            })
                                   
          except Exception as e:
            print(f"error {datum}: {e}")         
      else:
        print(f"no data {datum}")
                        
    except Exception as e:
      print(f"error: {e}")
    
    startDate += timedelta(days=1)

  print("KijkcijferData opgehaald")
  df = pd.DataFrame(kijkcijfersData)

  return df

def ophalenWeerData(startDate, endDate):
    latitude = 51.05
    longitude = 3.7167
    today = datetime.today().date()

    hourly_vars = [
        "temperature_2m", "apparent_temperature", "weather_code", "precipitation",
        "rain", "snowfall", "cloud_cover", "windspeed_10m", "sunshine_duration"
    ]
    
    common_params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": hourly_vars,
        "timezone": "Europe/Brussels",
        "temperature_unit": "celsius",
        "precipitation_unit": "mm",
        "windspeed_unit": "kmh"
    }

    def fetch_weather_data(api_url, start, end):
        params = common_params.copy()
        params.update({
            "start_date": start.strftime('%Y-%m-%d'),
            "end_date": end.strftime('%Y-%m-%d')
        })
        # print(f"Ophalen weer-data van {startDate.year}-{startDate.month}-{startDate.day} tot {endDate.year}-{endDate.month}-{endDate.day}")
        response = requests.get(api_url, params=params)
        if response.status_code == 200:
            data = response.json().get("hourly", {})
            df = pd.DataFrame({var: data.get(var, []) for var in hourly_vars})
            df["timestamp"] = pd.to_datetime(data.get("time", []))
            if not df.empty:
                df["hour"] = df["timestamp"].dt.hour
                df["day_of_week"] = df["timestamp"].dt.dayofweek
                df["month"] = df["timestamp"].dt.month
                df["year"] = df["timestamp"].dt.year
            return df
        else:
            print(f"Fout bij ophalen data: {response.status_code}")
            print(response.text)
            return pd.DataFrame()

    dataframes = []

    # Historische data
    if startDate.date() < today:
        print("Ophalen historische data")
        hist_end = min(endDate.date(), today - timedelta(days=1))
        dataframes.append(fetch_weather_data(
            "https://archive-api.open-meteo.com/v1/archive",
            startDate, datetime.combine(hist_end, datetime.min.time())
        ))

    # Forecast data
    if endDate.date() >= today:
        print("Ophalen forecast data")
        forecast_start = max(endDate, datetime.combine(today, datetime.min.time()))
        print(forecast_start)
        dataframes.append(fetch_weather_data(
            "https://api.open-meteo.com/v1/forecast",
            forecast_start, endDate
        ))

    if dataframes:
        print("Weerdata opgehaald")
        return pd.concat(dataframes).sort_values("timestamp").reset_index(drop=True)
    else:
        return pd.DataFrame()


In [54]:
teVoorspellen = pd.DataFrame({
  'dateDiff' : ['2025-04-22T00:00:00.000000', '2025-04-22T00:00:00.000000'],
  'ranking': ['13', '234'],
  'description': ['THUIS', 'HET 7 UUR-JOURNAAL'],
  'channel': ['VRT', 'EEN'],
  'startTime': ['18:00:00','19:00:00'],
  'rLength': ['00:30:00', '00:45:00'],
  'live': [0,0]
})

teVoorspellen['dateDiff'] = pd.to_datetime(teVoorspellen['dateDiff'])

end_date = teVoorspellen['dateDiff'].max()
start_date = end_date - timedelta(weeks=3)
histKijkcijfers = ophalenKijkcijferData(start_date, end_date - timedelta(days=1))
histWeerdata = ophalenWeerData(start_date, end_date)
histWeerdata

Ophalen kijkcijfer-data van 2025-4-1 tot 2025-4-21
KijkcijferData opgehaald
Ophalen historische data
Weerdata opgehaald


,temperature_2m,apparent_temperature,weather_code,precipitation,rain,snowfall,cloud_cover,windspeed_10m,sunshine_duration,timestamp,hour,day_of_week,month,year
0,5.6,2.5,0,0.0,0.0,0.0,0,10.7,0.00,2025-04-01 00:00:00,0,1,4,2025
1,5.2,2.2,0,0.0,0.0,0.0,0,10.1,0.00,2025-04-01 01:00:00,1,1,4,2025
2,4.8,1.8,0,0.0,0.0,0.0,0,9.8,0.00,2025-04-01 02:00:00,2,1,4,2025
3,4.6,1.3,0,0.0,0.0,0.0,0,10.5,0.00,2025-04-01 03:00:00,3,1,4,2025
4,4.1,1.0,0,0.0,0.0,0.0,0,9.6,0.00,2025-04-01 04:00:00,4,1,4,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
523,14.0,12.2,51,0.1,0.1,0.0,91,9.3,3600.00,2025-04-22 19:00:00,19,1,4,2025
524,13.0,11.8,3,0.0,0.0,0.0,100,5.4,3600.00,2025-04-22 20:00:00,20,1,4,2025
525,11.6,11.1,3,0.0,0.0,0.0,100,2.5,10.16,2025-04-22 21:00:00,21,1,4,2025
526,10.9,10.2,3,0.0,0.0,0.0,100,3.4,0.00,2025-04-22 22:00:00,22,1,4,2025


In [111]:
histKijkcijfers 

,dateDiff,ranking,description,channel,startTime,rLength,rateInK,live,Kijkers,date
0,2025-04-01T00:00:00.000000,1,THUIS,VRT 1,20:18:54,00:26:21,1.074.137,28,1074137,2025-04-01
1,2025-04-01T00:00:00.000000,2,FACTCHECKERS,VRT 1,20:47:19,00:47:10,933.174,28,933174,2025-04-01
2,2025-04-01T00:00:00.000000,3,HET 7 UUR-JOURNAAL,VRT 1,19:00:05,00:45:07,858.065,28,858065,2025-04-01
3,2025-04-01T00:00:00.000000,4,MAN BIJT HOND,VRT 1,19:48:04,00:23:00,728.369,28,728369,2025-04-01
4,2025-04-01T00:00:00.000000,5,FAMILIE,VTM,20:10:42,00:24:33,658.377,28,658377,2025-04-01
...,...,...,...,...,...,...,...,...,...,...
415,2025-04-21T00:00:00.000000,16,HUIZENJAGERS,PLAY4,22:30:10,00:52:02,258.883,7,258883,2025-04-21
416,2025-04-21T00:00:00.000000,17,DE AFSPRAAK,VRT CANVAS,20:32:38,00:47:13,233.113,7,233113,2025-04-21
417,2025-04-21T00:00:00.000000,18,ONDERZOEKSRECHTERS,VTM,21:53:05,00:46:33,208.055,7,208055,2025-04-21
418,2025-04-21T00:00:00.000000,19,HET JOURNAAL LAAT,VRT 1,22:24:15,00:18:59,205.777,7,205777,2025-04-21


## Cleaning data

In [55]:
def cleanKijkcijferData(df):
    # Zet 'Kijkers' kolom, als 'rateInK' bestaat
    if 'rateInK' in df.columns:
        df['Kijkers'] = (
            df['rateInK']
            .dropna()
            .astype(str)
            .str.replace('.', '', regex=False)
            .astype(int)
        )
    else:
        df['Kijkers'] = None

    # Tijd aanpassen
    tijd_regex = r'^\d{2}:\d{2}:\d{2}$'
    # Omzetten naar datetime
    df['date'] = pd.to_datetime(df['dateDiff']).dt.date
    
    # Filter rijen met formaat
    df = df[df['startTime'].str.match(tijd_regex, na=False) & df['rLength'].str.match(tijd_regex, na=False)].copy()
    
    # Afleveringlengte naar seconden omzetten 
    df['Lengte_sec'] = pd.to_timedelta(df['rLength']).dt.total_seconds().astype(int)
    
    # Uren met 24+
    def time_cor(rij):
        tijdArr = rij['startTime'].split(':')
        if int(tijdArr[0]) >= 24:
            tijdArr[0] = str(int(tijdArr[0]) - 24).zfill(2)
            rij['date'] += timedelta(days=1)
        rij['startTime'] = ':'.join(tijdArr)
        return rij
    
    df = df.apply(time_cor, axis=1)

    # 1 kolom voor beide data
    df['FullDate'] = pd.to_datetime(df['date'].astype(str) 
                                    + " " + df['startTime'].astype(str))
    
    # Hour en minute voor join later on
    df['hour'] = pd.to_datetime(df['startTime'], format='%H:%M:%S').dt.hour
    df['minute'] = 0

    # Kolommen verwijderen die niet nodig meer zijn, als ze bestaan
    columns_to_drop = ['startTime', 'rLength', 'rateInK', 'ranking', 'live']
    df.drop([col for col in columns_to_drop if col in df.columns], axis=1, inplace=True)

    # De nieuwe dataframe
    df = df[['FullDate', 'date', 'hour', 'minute', 'channel', 'description', 'Lengte_sec', 'Kijkers']]

    # Hernoemen kolommen
    df.rename(columns={'description': 'Programma', 'channel': 'Kanaal'}, inplace=True)

    return df


def cleanWeerData(df):
  weerData = df
  weerData['timestamp'] = pd.to_datetime(weerData['timestamp'])
  #naar zelfde formaat als kijkcijfer datum
  weerData['datetime'] = weerData['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

  #hour voor join later on
  weerData['hour'] = pd.to_datetime(weerData['datetime']).dt.hour
  weerData['minute'] = pd.to_datetime(weerData['datetime']).dt.minute
  weerData['date'] = pd.to_datetime(weerData['datetime']).dt.date

  #verwijder kolom
  weerData = weerData.drop(columns=['timestamp'])

  weerData = weerData[['datetime', 'date' ,'hour', 'minute', 'temperature_2m', 'apparent_temperature', 
                            'rain', 'snowfall', 'weather_code', 'cloud_cover', 
                            'windspeed_10m', 'sunshine_duration']]

  #hernoemen kolommen
  weerData.rename(columns={'temperature_2m':'Temperatuur', 'apparent_temperature':'Gevoelstemp', 'windspeed_10m': 'Windsnelheid', 'rain':'Regen', 'snowfall': 'Sneeuw', 'weather_code':'Weercode', 'cloud_cover':'Bewolking', 'sunshine_duration':'Zonnenschijn'}, inplace=True)

  return weerData

def mergen(kijkcijfers, weer):
  kijkcijfersWeer = pd.merge(kijkcijfers, weer, on=['date', 'hour'], how='left')
  kijkcijfersWeer = kijkcijfersWeer[['FullDate', 'date', 'hour', 'Kanaal', 'Programma', 'Lengte_sec', 'Kijkers', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Sneeuw', 'Weercode', 'Bewolking', 'Windsnelheid', 'Zonnenschijn']]
  kijkcijfersWeer.dropna(inplace=True)
  return kijkcijfersWeer

In [100]:
histWeerdataClean = cleanWeerData(histWeerdata)
histKijkcijfersClean = cleanKijkcijferData(histKijkcijfers)
# Merge historische kijkcijfers en weerdata
histKijkcijfersWeer = mergen(histKijkcijfersClean, histWeerdataClean)
histKijkcijfersWeer.dropna(inplace=True)
# te voorspellen data en weerdata mergen
teVoorspellenClean = cleanKijkcijferData(teVoorspellen)
teVoorspellenClean = teVoorspellenClean.drop(columns=['minute'])
teVoorspellenData = pd.merge(teVoorspellenClean, histWeerdataClean, on=['date', 'hour'], how='left')
teVoorspellenData = teVoorspellenData.drop(columns=['datetime', 'minute'])
#print("Aantal rijen in histKijkcijferWeerDf:", len(histKijkcijfersWeer))
#print(histKijkcijfersWeer.head()) 
teVoorspellenData

,FullDate,date,hour,Kanaal,Programma,Lengte_sec,Kijkers,Temperatuur,Gevoelstemp,Regen,Sneeuw,Weercode,Bewolking,Windsnelheid,Zonnenschijn
0,2025-04-22 18:00:00,2025-04-22,18,VRT,THUIS,1800,None,15.3,13.1,0.0,0.0,2,70,9.4,3600.0
1,2025-04-22 19:00:00,2025-04-22,19,EEN,HET 7 UUR-JOURNAAL,2700,None,14.0,12.2,0.1,0.0,51,91,9.3,3600.0


## OneHotEncoding

In [90]:
def oneHot(df):
  with open('./models/oneHotEncoder.pkl', 'rb') as oneHotFile:
    oneHotEnc = pickle.load(oneHotFile)

  lageKard = df[[ 'hour','Kanaal', 'isFeestdag', 'Weekdag', 'Seizoen']]
  dfOneHot = oneHotEnc.transform(lageKard)

  oneHotOutp = pd.DataFrame(dfOneHot.toarray(), 
                            columns=oneHotEnc.get_feature_names_out(), 
                            index=lageKard.index)

  df = df.drop(columns=['hour', 'Kanaal', 'isFeestdag', 'Weekdag', 'Seizoen'])
  df = pd.concat([df, oneHotOutp], axis = 1)
  return df

## TargetEncoding

In [91]:
def target(df):
  #target encoding voor medium kardinaliteiten
  with open('./models/oneHotTarget.pkl', 'rb') as f:
    targetEnc = pickle.load(f)
  medKardinaliteit = df[['date', 'Programma', 'Lengte_sec', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Bewolking', 'Windsnelheid', 'Zonnenschijn']]
  #verdere feature engineering op vorig model
  target = targetEnc.fit_transform(medKardinaliteit, df['Programma'])
  df = df.drop(columns=['date', 'Programma', 'Lengte_sec', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Bewolking', 'Windsnelheid', 'Zonnenschijn'])
  f = pd.concat([df, target], axis=1)

  return f


## Feature engineering

In [92]:
def tijdFeatures(df):
    cleanData = df
    cleanData['date'] = pd.to_datetime(cleanData['date'])
    #feestdagen
    feestdagen = holidays.BE()
    cleanData['isFeestdag'] = cleanData['date'].apply(lambda x: 1 if x in feestdagen else 0)
    #dag van de week
    cleanData['Weekdag'] = cleanData['date'].dt.weekday
    #weekend
    cleanData['isWeekend'] = cleanData['Weekdag'].apply(lambda x: 1 if x >= 5 else 0)
    #seizoenen
    cleanData['Seizoen'] = cleanData['date'].apply(seizoenFinder)

    return cleanData

#seizoen
def seizoenFinder(datum):
    inputDatum = datum.date()
    Y = inputDatum.year
    seizoenen = {
        'lente': (date(Y, 3, 20), date(Y, 6, 20)),
        'zomer': (date(Y, 6, 21), date(Y, 9, 22)),
        'herfst':   (date(Y, 9, 23), date(Y, 12, 20)),
        'winter': (date(Y, 12, 21), date(Y + 1, 3, 19)),
    }

    for seizoen, (start, end) in seizoenen.items():
        if start <= inputDatum <= end:
            return seizoen
    return 'winter'

def createLag(df, n):
  for i in range(1,n+1):
    df[f'KijkersLag{i}'] = df.sort_values('FullDate').groupby('Programma')['Kijkers'].shift(i).ffill()
  return df

def lagFeatures(predDf, histKijkcijferWeerDf):
  predDf['Kijkers'] = np.nan

  for i in range(1, 4):
     predDf[f'KijkersLag{i}'] = predDf.sort_values('FullDate').groupby(['Programma'])['Kijkers'].shift(i)
     predDf[f'KijkersLag{i}'] = predDf[f'KijkersLag{i}'].fillna(predDf.groupby(['Programma'])['Kijkers'].transform('mean'))
     
  predDf = pd.concat([histKijkcijferWeerDf, predDf], ignore_index=True)
  return predDf   


## Pipeline

In [94]:
def voorbereiding(toPredictData):
    # tijd features toevoegen
    tijdFeatures(toPredictData)
    tijdFeatures(histKijkcijfersWeer)
    print(toPredictData.columns)
    print(histKijkcijfersWeer.columns)

    # lag features toevoegen
    pred_hist_df = lagFeatures(toPredictData, histKijkcijfersWeer)

    # te voorspellen data er terug uithalen
    toPredictData = pred_hist_df[pred_hist_df['Kijkers'].isnull()]
  

    # One hot encoding
    toPredictData = oneHot(toPredictData)

    # Target encoding
    targetOneHotEnc = target(toPredictData)

    # numeric columns selecteren
    toPredictNumeric = targetOneHotEnc.select_dtypes(include=[np.number])
    
    return toPredictNumeric

In [101]:
data = voorbereiding(teVoorspellenData)
data.columns

Index(['FullDate', 'date', 'hour', 'Kanaal', 'Programma', 'Lengte_sec',
       'Kijkers', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Sneeuw', 'Weercode',
       'Bewolking', 'Windsnelheid', 'Zonnenschijn', 'isFeestdag', 'Weekdag',
       'isWeekend', 'Seizoen'],
      dtype='object')
Index(['FullDate', 'date', 'hour', 'Kanaal', 'Programma', 'Lengte_sec',
       'Kijkers', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Sneeuw', 'Weercode',
       'Bewolking', 'Windsnelheid', 'Zonnenschijn', 'isFeestdag', 'Weekdag',
       'isWeekend', 'Seizoen'],
      dtype='object')


Index(['Kijkers', 'Sneeuw', 'Weercode', 'isWeekend', 'KijkersLag1',
       'KijkersLag2', 'KijkersLag3', 'hour_0', 'hour_1', 'hour_2', 'hour_6',
       'hour_7', 'hour_8', 'hour_9', 'hour_10', 'hour_11', 'hour_12',
       'hour_13', 'hour_14', 'hour_15', 'hour_16', 'hour_17', 'hour_18',
       'hour_19', 'hour_20', 'hour_21', 'hour_22', 'hour_23', 'Kanaal_AB3',
       'Kanaal_CANVAS', 'Kanaal_CAZ', 'Kanaal_Canvas',
       'Kanaal_DAZN PRO LEAGUE 1 (NL)', 'Kanaal_EEN',
       'Kanaal_ELEVEN PRO LEAGUE 1 NL', 'Kanaal_EUROSPORT 1 (NL)',
       'Kanaal_KETNET', 'Kanaal_LA UNE', 'Kanaal_OP 12',
       'Kanaal_PLAY SPORTS OPEN', 'Kanaal_PLAY4', 'Kanaal_PLAY5',
       'Kanaal_PLAY6', 'Kanaal_Q2', 'Kanaal_RTL-TVI', 'Kanaal_TF1',
       'Kanaal_VIER', 'Kanaal_VIJF', 'Kanaal_VITAYA', 'Kanaal_VRT 1',
       'Kanaal_VRT CANVAS', 'Kanaal_VTM', 'Kanaal_VTM GOLD', 'Kanaal_VTM2',
       'Kanaal_VTM3', 'Kanaal_VTM4', 'Kanaal_ZES', 'isFeestdag_0',
       'isFeestdag_1', 'Weekdag_0', 'Weekdag_1', 'Weekda

## Voorspelling maken

In [107]:
def voorspellingMaken(data):
  transformer = FunctionTransformer(voorbereiding)

  voorbereidingPipeline = Pipeline([
        ('voorbereiding', transformer),
        ('standardScaler', StandardScaler())
    ])

  preprocessed_data = voorbereidingPipeline.fit_transform(data)

  with open('./models/lightGBM.pkl', 'rb') as file:
      lightgbm = pickle.load(file)

  predictions = lightgbm.predict(preprocessed_data)
  
  return predictions

prediction = voorspellingMaken(teVoorspellenData)

Index(['FullDate', 'date', 'hour', 'Kanaal', 'Programma', 'Lengte_sec',
       'Kijkers', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Sneeuw', 'Weercode',
       'Bewolking', 'Windsnelheid', 'Zonnenschijn', 'isFeestdag', 'Weekdag',
       'isWeekend', 'Seizoen', 'KijkersLag1', 'KijkersLag2', 'KijkersLag3'],
      dtype='object')
Index(['FullDate', 'date', 'hour', 'Kanaal', 'Programma', 'Lengte_sec',
       'Kijkers', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Sneeuw', 'Weercode',
       'Bewolking', 'Windsnelheid', 'Zonnenschijn', 'isFeestdag', 'Weekdag',
       'isWeekend', 'Seizoen'],
      dtype='object')


c:\Users\krist\Documents\Hogent_IT\2de_jaar\Machine_Learning\ML_project_kijkcijfers\.venv\Lib\site-packages\sklearn\utils\extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
c:\Users\krist\Documents\Hogent_IT\2de_jaar\Machine_Learning\ML_project_kijkcijfers\.venv\Lib\site-packages\sklearn\utils\extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
c:\Users\krist\Documents\Hogent_IT\2de_jaar\Machine_Learning\ML_project_kijkcijfers\.venv\Lib\site-packages\sklearn\utils\extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
c:\Users\krist\Documents\Hogent_IT\2de_jaar\Machine_Learning\ML_project_kijkcijfers\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Prediction resultaten

In [108]:
prediction

array([91695.96511719, 63424.95262843])

In [110]:
testData = pd.read_csv('./data csv/')

X_test = testData.drop(columns=['rateInk'])
y_test = testData['rateInk'].apply(lambda x: int(''.join(x.split('.'))))

predictions = voorspellingMaken(X_test)

# mae en mape berekenen
mae = mean_absolute_error(y_test, predictions)
mape = mean_absolute_percentage_error(y_test, predictions)
print(f"MAE: {mae}")
print(f"MAPE: {mape}")

resultaten = pd.DataFrame({
    'Actual': y_test,
    'Predicted': predictions
})

resultaten['Difference'] = resultaten['Actual'] - resultaten['Predicted']
resultaten['Percentage Difference'] = (resultaten['Difference'] / resultaten['Actual']) * 100

resultaten.head(10)

PermissionError: [Errno 13] Permission denied: './data csv/'

In [ ]:
resultaten.describe()